# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# 24 · Training coverage and observed motion

**No new models. No additional evaluation scoring.** Round 10 missed its continuation gate. This milestone checks how much of the frozen training-game data the small diagnostic subset uses, then constructs 12 observed-only channels. The new representation is not connected to either existing model and has no measured accuracy gain.

In [ ]:
from pathlib import Path
import json, sys, subprocess, signal
import plotly.io as pio
KIT = Path('/home/sagemaker-user/nfl_feature_round11')
OUT = Path('/home/sagemaker-user/nfl-feature-round11-results')
PY = Path('/home/sagemaker-user/nfl-player-trajectory/.venv/bin/python')
if not KIT.is_dir() or not PY.is_file():
    raise FileNotFoundError('Use the existing NFL space and extract this kit first.')
sys.path.insert(0, str(KIT))
import visuals
pio.renderers.default = 'plotly_mimetype'
def run(stage):
    p = subprocess.Popen([str(PY),str(KIT/'run_round.py'),stage], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,bufsize=1)
    try:
        for line in p.stdout: print(line,end='')
        code=p.wait()
    except KeyboardInterrupt:
        p.send_signal(signal.SIGINT)
        try:p.wait(timeout=10)
        except subprocess.TimeoutExpired:p.kill();p.wait()
        raise
    if code:
        raise RuntimeError(f'{stage} stopped ({code}). Export the report; do not loosen gates or reinstall packages.')
def show(fig,name):
    visuals.save(fig,OUT,name).show()


In [ ]:
show(visuals.previous_metrics(KIT),'round10_metrics')
show(visuals.previous_contrast(KIT),'round10_contrast')

## 1. Verify the completed two-arm parent
This package requires the **19,250-parameter Cartesian/goal** Round 10 version. Source and receipt checks reject the incompatible three-arm variant. No source is changed.

In [ ]:
run('preflight')

## 2. Count unused observed plays in the same training games
Only game/play identity columns are parsed from existing verified input CSVs. No unused output labels are inspected, no validation coordinates are loaded, and no new selection is made.

In [ ]:
run('coverage')
show(visuals.coverage(OUT),'training_coverage')

## 3. Test 32 training plays first
Six channels provide direct receiver-relative geometry/motion. Six describe adjacent-frame acceleration, turning and speed change. Missing frames are not bridged. These overlap concepts already considered elsewhere; they are not new raw signals.

In [ ]:
run('smoke')

## 4. Prepare only existing training plays
Continue only after `motion_receiver_smoke_passed`. Previous feature checkpoints are retained and verified. Targets are not unpacked in this stage. Quantiles and tail magnitudes are training-input diagnostics, not automatic clipping rules.

In [ ]:
run('features')
show(visuals.support(OUT),'candidate_support')
show(visuals.tails(OUT),'candidate_tails')

Save this notebook. Continue to notebook 25 for frozen final-checkpoint training-error measurement. Do not train the new candidates yet.